# Task 2: Exploratory Data Analysis
**Project:** Forecasting Financial Inclusion in Ethiopia  
**Analyst:** Selam Analytics  
**Date:** 2026-07-16

## Objective
Analyse patterns and factors influencing financial inclusion in Ethiopia. Document at least 5 key insights with supporting evidence.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
COLORS = sns.color_palette('Set2', 10)
ACCENT = '#2196F3'
HIGHLIGHT = '#FF5722'

df = pd.read_csv('../data/processed/ethiopia_fi_enriched.csv', parse_dates=['observation_date'])
obs = df[df['record_type'] == 'observation'].copy()
events = df[df['record_type'] == 'event'].copy()
events['observation_date'] = pd.to_datetime(events['observation_date'])
links = df[df['record_type'] == 'impact_link'].copy()
targets = df[df['record_type'] == 'target'].copy()

print('Data loaded. Shape:', df.shape)

## 2.1 Access Analysis: Account Ownership Trajectory

In [ ]:
# Account ownership trajectory (Findex years only)
acc = obs[obs['indicator_code'] == 'ACC_OWNERSHIP'].sort_values('observation_date')

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel 1: Trajectory with growth annotations
ax = axes[0]
ax.plot(acc['observation_date'].dt.year, acc['value_numeric'], 
        'o-', color=ACCENT, linewidth=2.5, markersize=10, label='Account Ownership')
ax.fill_between(acc['observation_date'].dt.year, acc['value_numeric'], alpha=0.15, color=ACCENT)

# Annotate data points
for _, row in acc.iterrows():
    ax.annotate(f"{row['value_numeric']:.0f}%",
                xy=(row['observation_date'].year, row['value_numeric']),
                xytext=(0, 12), textcoords='offset points', ha='center', fontweight='bold', fontsize=11)

# Target line
ax.axhline(70, color=HIGHLIGHT, linestyle='--', linewidth=1.5, label='NFIS-II Target 2030 (70%)')
ax.axhline(55, color='orange', linestyle=':', linewidth=1.5, label='NFIS-II Target 2025 (55%)')

ax.set_title('Ethiopia Account Ownership Rate\n(Global Findex Surveys 2011–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Share of Adults (%)')
ax.set_ylim(0, 85)
ax.legend(loc='upper left', fontsize=9)
ax.set_xticks(acc['observation_date'].dt.year)

# Panel 2: Growth rates between surveys
ax2 = axes[1]
years = acc['observation_date'].dt.year.values
values = acc['value_numeric'].values
growth = np.diff(values)
periods = [f"{years[i]}-{years[i+1]}" for i in range(len(years)-1)]
colors = [COLORS[0] if g > 0 else HIGHLIGHT for g in growth]
bars = ax2.bar(periods, growth, color=colors, edgecolor='white', linewidth=1)
for bar, val in zip(bars, growth):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
             f'+{val:.0f}pp', ha='center', fontweight='bold', fontsize=12)

ax2.set_title('Growth in Account Ownership Between Surveys\n(percentage points)', fontsize=13, fontweight='bold')
ax2.set_xlabel('Survey Period')
ax2.set_ylabel('Change (pp)')
ax2.set_ylim(0, 20)
ax2.axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig('../reports/figures/task2_access_trajectory.png', dpi=150, bbox_inches='tight')
plt.show()

### Key Insight 1: The 2021–2024 Slowdown

Despite **54 million Telebirr registrations** and **10 million M-Pesa registrations**, account ownership grew only **+3pp** (2021–2024) compared to +11pp in 2017–2021. This suggests:

1. **Registered ≠ Active**: Most Telebirr registrations may represent inactive or duplicate accounts
2. **Already-banked users**: Many Telebirr users likely already had bank accounts — the Findex measures unique individual account holders, not unique registered accounts
3. **Penetration ceiling**: The easiest-to-reach population may already have accounts; remaining unbanked are harder to reach (rural, elderly, low-literacy)
4. **Survey vs operator definitions differ**: Telebirr counts SIM-linked registrations; Findex counts active account users in past 12 months

In [ ]:
# Illustrate the registered vs active gap
fig, ax = plt.subplots(figsize=(10, 5))

# Ethiopian adult population ~65 million
pop_adults = 65

categories = ['Telebirr\nRegistered', 'M-Pesa\nRegistered', 'Total Operator\nRegistered', 
              'Findex Mobile Money\n(Active, 2024)']
values_millions = [54, 10, 64, pop_adults * 0.0945]  # 9.45% of 65M adults
colors_bar = [COLORS[0], COLORS[1], COLORS[2], HIGHLIGHT]

bars = ax.bar(categories, values_millions, color=colors_bar, edgecolor='white', linewidth=1.5, width=0.5)
for bar, val in zip(bars, values_millions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{val:.1f}M', ha='center', fontweight='bold', fontsize=12)

ax.set_title('The Registration–Activity Gap\nOperator-Registered Users vs Findex Active Mobile Money Users (2024)', 
             fontsize=13, fontweight='bold')
ax.set_ylabel('Users / Account Holders (millions)')
ax.set_ylim(0, 75)
ax.axhline(pop_adults * 0.0945, color=HIGHLIGHT, linestyle='--', linewidth=1.5, alpha=0.7)
ax.text(3.5, pop_adults * 0.0945 + 1, 'Findex active baseline', color=HIGHLIGHT, ha='right', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/task2_registration_gap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: ~6.1M adults are Findex-active mobile money users vs 64M+ operator registrations — a ~10:1 ratio.')

## 2.2 Gender Gap Analysis

In [ ]:
# Gender gap in access
female_acc = obs[obs['indicator_code'] == 'ACC_OWNERSHIP_FEMALE'].sort_values('observation_date')
male_acc = obs[obs['indicator_code'] == 'ACC_OWNERSHIP_MALE'].sort_values('observation_date')
female_pay = obs[obs['indicator_code'] == 'USG_DIGITAL_PAYMENT_FEMALE']
male_pay = obs[obs['indicator_code'] == 'USG_DIGITAL_PAYMENT_MALE']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Account ownership gender gap
ax = axes[0]
years_gender = [2021, 2024]
female_vals = female_acc['value_numeric'].values
male_vals = male_acc['value_numeric'].values

x = np.arange(len(years_gender))
width = 0.35
ax.bar(x - width/2, female_vals, width, label='Female', color=COLORS[3], edgecolor='white')
ax.bar(x + width/2, male_vals, width, label='Male', color=COLORS[0], edgecolor='white')

for i, (f, m) in enumerate(zip(female_vals, male_vals)):
    ax.text(i - width/2, f + 0.5, f'{f:.0f}%', ha='center', fontweight='bold', fontsize=11)
    ax.text(i + width/2, m + 0.5, f'{m:.0f}%', ha='center', fontweight='bold', fontsize=11)
    gap = m - f
    ax.annotate(f'Gap: {gap:.0f}pp', xy=(i, max(f, m) + 3), ha='center', fontsize=10, 
                color=HIGHLIGHT, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(years_gender)
ax.set_title('Account Ownership by Gender', fontsize=13, fontweight='bold')
ax.set_ylabel('Share of Adults (%)')
ax.set_ylim(0, 75)
ax.legend()

# Digital payment gender gap
ax2 = axes[1]
categories_pay = ['Digital Payment\n(Female)', 'Digital Payment\n(Male)']
pay_vals = [float(female_pay['value_numeric'].iloc[0]), float(male_pay['value_numeric'].iloc[0])]
bars = ax2.bar(categories_pay, pay_vals, color=[COLORS[3], COLORS[0]], edgecolor='white', width=0.4)
for bar, val in zip(bars, pay_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.0f}%', 
             ha='center', fontweight='bold', fontsize=12)

ax2.set_title('Digital Payment Usage by Gender (2024)', fontsize=13, fontweight='bold')
ax2.set_ylabel('Share of Adults (%)')
ax2.set_ylim(0, 55)
gap_pay = pay_vals[1] - pay_vals[0]
ax2.text(0.5, 47, f'Gender gap: {gap_pay:.0f}pp', ha='center', transform=ax2.transAxes,
         fontsize=12, color=HIGHLIGHT, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/figures/task2_gender_gap.png', dpi=150, bbox_inches='tight')
plt.show()

### Key Insight 2: Narrowing Gender Gap in Access, but Persistent in Usage

- Account ownership gender gap **narrowed** from 20pp (2021: Male 56%, Female 36%) to **10pp** (2024: Male 54%, Female 44%)
- Male ownership actually **fell slightly** (56%→54%) while female rose sharply (36%→44%) — suggesting Telebirr/M-Pesa drew in primarily female first-time account holders
- **Digital payment gender gap remains large at 14pp** (Male 42%, Female 28%) — women have accounts but use them less for digital payments

## 2.3 Urban–Rural Analysis

In [ ]:
urban = float(obs[obs['indicator_code'] == 'ACC_OWNERSHIP_URBAN']['value_numeric'].iloc[0])
rural = float(obs[obs['indicator_code'] == 'ACC_OWNERSHIP_RURAL']['value_numeric'].iloc[0])
overall = float(obs[(obs['indicator_code'] == 'ACC_OWNERSHIP') & 
                    (obs['observation_date'].dt.year == 2024)]['value_numeric'].iloc[0])

fig, ax = plt.subplots(figsize=(8, 5))
categories = ['Urban', 'National Average', 'Rural']
values = [urban, overall, rural]
colors = [COLORS[0], COLORS[2], COLORS[1]]
bars = ax.bar(categories, values, color=colors, edgecolor='white', linewidth=1.5, width=0.4)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.5, f'{val:.0f}%', 
            ha='center', fontweight='bold', fontsize=13)
ax.set_title('Account Ownership by Location (2024)', fontsize=14, fontweight='bold')
ax.set_ylabel('Share of Adults (%)')
ax.set_ylim(0, 80)
ax.text(0.5, 0.92, f'Urban-Rural Gap: {urban - rural:.0f}pp', transform=ax.transAxes, 
        ha='center', fontsize=12, color=HIGHLIGHT, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/task2_urban_rural.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Urban-Rural gap: {urban - rural:.0f}pp')
print('Note: ~80% of Ethiopians live in rural areas — the rural gap is the primary inclusion challenge.')

## 2.4 Event Timeline and Indicator Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

# Access trajectory
ax.plot(acc['observation_date'].dt.year, acc['value_numeric'], 
        'o-', color=ACCENT, linewidth=2.5, markersize=10, label='Account Ownership (%)', zorder=5)

# Mobile money trajectory
mm = obs[obs['indicator_code'] == 'ACC_MM_ACCOUNT'].sort_values('observation_date')
ax.plot(mm['observation_date'].dt.year, mm['value_numeric'], 
        's--', color=COLORS[1], linewidth=2, markersize=8, label='Mobile Money Account (%)', zorder=5)

# Event markers
event_annotations = [
    (2020.5, 'Agent Banking\nDirective', COLORS[4]),
    (2021.0, 'Telebirr\nLaunch', HIGHLIGHT),
    (2021.5, 'NFIS-II\nLaunch', COLORS[5]),
    (2022.0, 'Fayda ID', COLORS[6]),
    (2022.5, 'EthSwitch\nInterop', COLORS[3]),
    (2022.7, 'Safaricom\nEntry', COLORS[7]),
    (2023.0, 'M-Pesa\nLaunch', COLORS[2]),
    (2023.5, 'P2P>ATM\nMilestone', COLORS[9]),
]

for yr, label, color in event_annotations:
    ax.axvline(yr, color=color, linestyle=':', linewidth=1.5, alpha=0.8)
    ax.text(yr, 72, label, ha='center', fontsize=7.5, color=color, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8, edgecolor=color))

# Target
ax.axhline(70, color=HIGHLIGHT, linestyle='--', linewidth=1.5, alpha=0.7, label='NFIS-II Target 2030 (70%)')
ax.axhline(55, color='orange', linestyle=':', linewidth=1.5, alpha=0.7, label='NFIS-II Interim 2025 (55%)')

ax.set_xlim(2010, 2025)
ax.set_ylim(0, 85)
ax.set_title('Ethiopia Financial Inclusion Indicators with Key Events (2011–2024)', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Share of Adults (%)')
ax.legend(loc='upper left', fontsize=9)

plt.tight_layout()
plt.savefig('../reports/figures/task2_event_timeline_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.5 Infrastructure and Enablers

In [ ]:
# Mobile penetration and 4G coverage trends
mobile = obs[obs['indicator_code'] == 'INFRA_MOBILE_PENETRATION'].sort_values('observation_date')
coverage_4g = obs[obs['indicator_code'] == 'INFRA_4G_COVERAGE'].sort_values('observation_date')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mobile penetration
ax = axes[0]
ax.plot(mobile['observation_date'].dt.year, mobile['value_numeric'], 
        'o-', color=COLORS[0], linewidth=2.5, markersize=10)
ax.fill_between(mobile['observation_date'].dt.year, mobile['value_numeric'], alpha=0.15, color=COLORS[0])
for _, row in mobile.iterrows():
    ax.text(row['observation_date'].year, row['value_numeric'] + 1.5,
            f"{row['value_numeric']:.0f}%", ha='center', fontweight='bold')
ax.set_title('Mobile Penetration Rate\n(SIM connections as share of population)', fontsize=12, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('%')
ax.set_ylim(0, 80)

# 4G coverage
ax2 = axes[1]
ax2.plot(coverage_4g['observation_date'].dt.year, coverage_4g['value_numeric'], 
         's-', color=COLORS[1], linewidth=2.5, markersize=10)
ax2.fill_between(coverage_4g['observation_date'].dt.year, coverage_4g['value_numeric'], 
                  alpha=0.15, color=COLORS[1])
for _, row in coverage_4g.iterrows():
    ax2.text(row['observation_date'].year, row['value_numeric'] + 1.5,
             f"{row['value_numeric']:.0f}%", ha='center', fontweight='bold')
ax2.set_title('4G Population Coverage\n(share of population with 4G access)', fontsize=12, fontweight='bold')
ax2.set_xlabel('Year')
ax2.set_ylabel('%')
ax2.set_ylim(0, 80)

plt.tight_layout()
plt.savefig('../reports/figures/task2_infrastructure.png', dpi=150, bbox_inches='tight')
plt.show()

## 2.6 Correlation Analysis

In [ ]:
# Build correlation matrix from available observations
# Pivot: each year as row, each indicator as column
obs_pivot = obs.copy()
obs_pivot['year'] = obs_pivot['observation_date'].dt.year

key_indicators = ['ACC_OWNERSHIP', 'ACC_MM_ACCOUNT', 'USG_DIGITAL_PAYMENT',
                  'INFRA_MOBILE_PENETRATION', 'INFRA_4G_COVERAGE', 'INFRA_AGENT_DENSITY']

pivot = obs_pivot[obs_pivot['indicator_code'].isin(key_indicators)].pivot_table(
    index='year', columns='indicator_code', values='value_numeric', aggfunc='mean'
)

corr = pivot.corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, ax=ax, cmap='RdYlGn', center=0, vmin=-1, vmax=1,
            annot=True, fmt='.2f', linewidths=0.5,
            cbar_kws={'label': 'Pearson Correlation'})
ax.set_title('Indicator Correlation Matrix\n(based on overlapping annual observations)', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/task2_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Note: Correlations based on very few overlapping data points — treat as directional, not definitive.')

## 2.7 Key Insights Summary

### Insight 1: The 2021–2024 Access Slowdown is Real and Explained by Definitional Gaps
Account ownership grew only +3pp (46%→49%) despite 65M+ operator registrations. The Findex measures *active* account holders using accounts in the past 12 months — not registered users. An estimated 6.1M adults are Findex-active mobile money users vs 64M+ operator registrations, a ~10:1 ratio. Ethiopia-specific nuance: mobile money-only users remain rare (~0.5% per market context); most mobile money users also have bank accounts.

### Insight 2: Gender Gap Narrowing in Access but Not Usage
The access gender gap halved from 20pp to 10pp (2021→2024), primarily because female account ownership rose 8pp while male ownership slightly fell. However, the digital payment gender gap (14pp) remains large, suggesting women gain accounts but don't transition to active digital payment behaviour — a critical policy target.

### Insight 3: Urban-Rural Gap is the Primary Access Challenge
With a 21pp urban-rural gap (Urban: 62%, Rural: 41%) and ~80% of Ethiopia's population living in rural areas, rural inclusion is the primary determinant of national headline figures. Mobile money and agent networks are the primary vehicle for rural inclusion.

### Insight 4: P2P Surpassing ATM Withdrawals Signals a Behavioural Shift, Not Just Registration
In 2023, P2P digital transfers (ETB 180B) exceeded ATM cash withdrawals (ETB 160B) for the first time. This is not a registration metric — it captures actual transaction behaviour. It suggests digital payment depth is growing even if Findex headline rates are slow to move.

### Insight 5: Infrastructure is the Binding Constraint for Rural Usage
4G coverage reached 68% of population by 2024 and agent density is expanding, but rural areas remain underserved. The strong positive correlation between mobile penetration and account ownership confirms that connectivity is the binding constraint for rural mobile money adoption.

### Insight 6: Telebirr's Scale Did Not Translate 1:1 into Findex Inclusion
Telebirr reached 54M users (nearly all Ethiopian adults who could register) but Findex mobile money ownership is 9.45%. This confirms the scale of inactive/duplicate registrations and the definitional gap between operator metrics and survey-based inclusion measurement. Policy targeting must use both datasets — operators for real-time tracking, Findex for impact evaluation.

### Insight 7: Market Competition is Beginning to Drive Usage
Safaricom's entry (Aug 2022) and M-Pesa launch (Aug 2023) introduced competition. Historical evidence from Kenya, Tanzania, and Ghana shows that multi-provider mobile money markets see 2–3x acceleration in active user growth vs monopoly markets. Ethiopia's 2025–2027 period should see this effect materialise.